# Storage-Aware Smoothing Metrics for Short-Term Estimated WEC Power

This notebook connects estimated wave energy converter (WEC) power variability, short-term forecasts, prediction intervals, and simple storage-aware smoothing metrics.

The goal is not to optimize BESS dispatch or produce final storage sizing. Instead, the notebook asks:

> Given a generic 250 kW WEC proxy, what storage burden is implied if the grid receives a smoother, forecast-informed, or uncertainty-aware version of the WEC power signal?

The analysis uses the 250 kW interpretation scale to express normalized estimated WEC-power results in approximate physical units. These values remain interpretation aids for the simplified WEC proxy, not final device or project ratings.


## 1. Scope, upstream workflow, and smoothing concept

This notebook builds on the previous workflow steps:

* [Literature map and research context](../outputs/pdf/00_literature_map.pdf)
* [Wave-resource data preparation](../outputs/pdf/01_wave_data_preparation.pdf)
* [Estimated WEC power from sea-state conditions](../outputs/pdf/02_wec_power_estimation.pdf)
* [Short-term point forecasting baselines](../outputs/pdf/03_forecasting_baselines.pdf)
* [Prediction intervals and uncertainty estimation](../outputs/pdf/04_prediction_intervals_uncertainty.pdf)
* Storage-aware smoothing metrics

The previous notebooks prepared the wave-resource data, estimated WEC power from sea-state conditions, evaluated short-term point forecasts, and calibrated empirical prediction intervals. This notebook uses those outputs to evaluate how different grid-export target definitions affect ramp-rate reduction and implied storage power, energy, throughput, and simple state-of-charge behavior.

The grid-export target is not prescribed as an arbitrary constant power level. Instead, it is derived from the estimated WEC power signal, point forecasts, or prediction interval bounds. This keeps the smoothing target tied to the available wave-energy resource. The analysis therefore asks how much storage power and energy would be required to transform a variable WEC power signal into a smoother grid-export profile.

The target represents a simplified export setpoint at the point of common coupling (PCC). It should not be interpreted as grid demand, market dispatch, or a validated grid-code requirement.


### Conceptual storage-smoothing balance

```text
Estimated WEC power
      p_wec
        │
        ▼
 ┌──────────────────┐
 │ Smoothing target │───▶ Grid export target
 │  selection rule  │        p_grid
 └──────────────────┘
        │
        ▼
Storage balancing power:
p_st = p_wec - p_grid

p_st > 0  → storage charges / absorbs surplus WEC power
p_st < 0  → storage discharges / supports grid delivery
```

In this notebook, storage is represented as an ideal balancing buffer connected at the grid-export interface. The WEC power is not assumed to pass entirely through storage. Instead, storage absorbs the positive mismatch when estimated WEC power exceeds the selected grid-export target and supplies the negative mismatch when estimated WEC power falls below that target.

The resulting storage power, cumulative energy swing, and state-of-charge trajectory are interpreted as smoothing-implied requirements, not as final BESS sizing results.


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown


pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True


# Paths are relative to the notebooks/ directory.
NOTEBOOK_ID = "notebook_05"

DATA_DIR = Path("../data")
OUTPUTS_DIR = Path("../outputs")

TABLES_DIR = OUTPUTS_DIR / "tables" / NOTEBOOK_ID
FIGURES_DIR = OUTPUTS_DIR / "figures" / NOTEBOOK_ID
NOTEBOOK_OUTPUT_DIR = OUTPUTS_DIR / NOTEBOOK_ID

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Main input files generated by previous notebooks.
WEC_POWER_PATH = DATA_DIR / "processed" / "leixoes_wec_power_30min_estimated.parquet"
FORECAST_PATH = OUTPUTS_DIR / "notebook_03" / "forecast_predictions.parquet"
INTERVAL_PATH = OUTPUTS_DIR / "notebook_04" / "interval_predictions.parquet"


# Main interpretation scale used throughout this notebook.
RATED_POWER_KW = 250.0

# Main forecast / uncertainty setup used for the storage-aware scenarios.
SELECTED_MODEL = "Ridge"
SELECTED_INTERVAL_METHOD = "ConformalStyle_ModelHorizon"
MAIN_INTERVAL_LEVEL = 0.90

# The input data are on a 30-minute time step.
DT_MINUTES = 30
DT_HOURS = DT_MINUTES / 60

SMOOTHING_WINDOWS_MIN = [30, 60, 120, 240]

SMOOTHING_WINDOW_LABELS = {
    30: "30 min",
    60: "1 h",
    120: "2 h",
    240: "4 h",
}

wec_df = pd.read_parquet(WEC_POWER_PATH)
forecast_df = pd.read_parquet(FORECAST_PATH)
interval_df = pd.read_parquet(INTERVAL_PATH)


# WEC power time series from Notebook 02
wec_df = wec_df[
    [
        "time",
        "wec_power_kw_estimated",
        "wec_power_norm_estimated",
    ]
].copy()

wec_df["time"] = pd.to_datetime(wec_df["time"])

# Alias used in this notebook to make the 250 kW interpretation scale explicit.
wec_df["wec_power_kw_250"] = wec_df["wec_power_kw_estimated"]

wec_df = (
    wec_df
    .dropna(subset=["wec_power_kw_250", "wec_power_norm_estimated"])
    .sort_values("time")
    .reset_index(drop=True)
)


# Point forecasts from Notebook 03
forecast_df = forecast_df[
    [
        "sample_id",
        "origin_time",
        "target_time",
        "horizon_steps",
        "horizon_hours",
        "horizon_label",
        "fold_id",
        "split",
        "model",
        "y_true_norm",
        "y_pred_norm",
        "error_norm",
        "y_true_kw_250",
        "y_pred_kw_250",
        "error_kw_250",
    ]
].copy()

forecast_df["origin_time"] = pd.to_datetime(forecast_df["origin_time"])
forecast_df["target_time"] = pd.to_datetime(forecast_df["target_time"])


# Prediction intervals from Notebook 04
interval_df = interval_df[
    [
        "sample_id",
        "origin_time",
        "target_time",
        "horizon_steps",
        "horizon_hours",
        "horizon_label",
        "fold_id",
        "split",
        "model",
        "interval_method",
        "interval_level",
        "y_true_norm",
        "y_pred_norm",
        "lower_norm",
        "upper_norm",
        "interval_width_norm",
        "y_true_kw_250",
        "y_pred_kw_250",
        "lower_kw_250",
        "upper_kw_250",
        "interval_width_kw_250",
        "covered",
    ]
].copy()

interval_df["origin_time"] = pd.to_datetime(interval_df["origin_time"])
interval_df["target_time"] = pd.to_datetime(interval_df["target_time"])


print(f"Loaded WEC power time series: {wec_df.shape[0]:,} rows")
print(f"Loaded forecast predictions: {forecast_df.shape[0]:,} rows")
print(f"Loaded interval predictions: {interval_df.shape[0]:,} rows")

Loaded WEC power time series: 6,220 rows
Loaded forecast predictions: 102,396 rows
Loaded interval predictions: 615,072 rows
